# Celestial Navigation with EqF

## First section - Process star images

Package installations

In [68]:
%pip install sympy matplotlib scipy astropy sep opencv-python pyproj -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
# take functions from other hw  ( replace them with SO(3) versions ) 
import numpy as np
def so3_wedge(w):
    w = w.flatten()
    return np.array([
        [0, -w[2], w[1]],
        [w[2], 0, -w[0]],
        [-w[1], w[0], 0]
    ])

def so3_exp(w):
    w = w.flatten()
    theta = np.linalg.norm(w)
    I = np.eye(3)
    
    if theta < 1e-7:
        return I
    
    w_hat = w / theta
    w_wedge = so3_wedge(w_hat)
    
    # Rodrigues' Rotation Formula
    R = I + np.sin(theta) * w_wedge + (1 - np.cos(theta)) * (w_wedge @ w_wedge)
    return R

def so3_vee(Phi):
    # Extracts [w1, w2, w3] from the skew-symmetric form
    return np.array([Phi[2, 1], Phi[0, 2], Phi[1, 0]])

def ad_so3(R):
    # For SO(3), Adjoint(R) is simply R
    return R
    
def so3_log(R):
    I = np.eye(3)
    cos_theta = np.clip((np.trace(R) - 1) / 2, -1.0, 1.0)
    theta = np.arccos(cos_theta)
    
    if abs(theta) < 1e-7:
        return np.zeros(3)
    
    # Extract the skew-symmetric part
    w_wedge = (theta / (2 * np.sin(theta))) * (R - R.T)
    return so3_vee(w_wedge)

Plate solve images from the video

In [ ]:
from astropy.wcs import WCS
from astropy.io import fits
from astropy.coordinates import SkyCoord
import astropy.units as units
import sep
import cv2
import os
import subprocess
import numpy as np

astap_path = "C:\\Program Files\\astap\\astap.exe"
video_path = "C:\\Users\\selen\\Documents\\puthon\\590\\CNV_videos\\videodata_0430.mov"
gravity_path = "C:\\Users\\selen\\Documents\\puthon\\590\\CNV_csvs\\Gravity.csv"
gyro_path = "C:\\Users\\selen\\Documents\\puthon\\590\\CNV_csvs\\Gyroscope.csv"
# solve the image 
def solve_img_local(imgpath):
    cmd = [astap_path, "-f", imgpath, "-z", "0"]
    subprocess.run(cmd)
    wcs_file = imgpath.replace(".jpg", ".wcs")
    return wcs_file

header = fits.getheader('solved_image.fits')
wcs = WCS(header)

# first get each frame of the video separately
# load video:
vid = 'starchart.mov'
cap = cv2.VideoCapture(vid)
# output directory
output_dir = 'frames'
os.makedirs(output_dir, exist_ok=True)

count = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break  # End of video
    
    # save each frame as a JPG
    frame_name = os.path.join(output_dir, f"frame_{count:04d}.jpg")
    cv2.imwrite(frame_name, frame)
    count += 1

cap.release()
print(f"Extracted {count} frames.")

# turn all the frames into 'data' and get coordinates
import glob
frame_files = sorted(glob.glob(os.path.join(output_dir, "*.jpg")))

for frame_path in frame_files:
    # load and convert
    img = cv2.imread(frame_path, cv2.IMREAD_GRAYSCALE)
    data = img.astype(np.float32)

    # analyze with SEP
    bkg = sep.Background(data)
    objects = sep.extract(data - bkg, 1.5, err=bkg.globalrms)

    # convert extracted XY cords into a list for WCS
    pixel_coords = np.column_stack((objects['x'], objects['y']))
    # world coordinates
    world_coords = wcs.all_pix2world(pixel_coords,1)


FileNotFoundError: [Errno 2] No such file or directory: 'solved_image.fits'

Define the EqF class

In [60]:
import numpy as np

class Equivariant:

    def __init__(self, dt, sigma_x, sigma_y, sigma_z, sigma_gravity):
        self.dt = dt

        # state (world frame)
        self.xi = np.array([0,0,1]) # state in world frame ?
        self.X = so3_exp(self.xi)

        self.xi0 = np.array([0,0,1]) 
        self.X0 = so3_exp(self.xi0)
    
        # initial condition for estimate
        self.ghat = np.array([0,0,1])
        # process noise covariance
        self.Q = np.eye(3) * sigma_gravity
        # sensor noise covariance
        self.Rcov = np.array([[sigma_x, 0,0],[0,sigma_y,0],[0,0,sigma_z]])

        self.Xhat = np.eye(3)
        self.Delta = np.eye(3)

        # initialize ricatti term
        self.sigma = np.eye(2) # represents the initial uncertainty (of calculated initial states)
        self.Cts = np.eye(3)

    # functions for easier computation: 
    def lift(self,in1, in2): # lift S2 element to lie algebra (go from g to xi)
        return so3_exp(in2)  
    def unlift(self,A): # lie algebra to S2 (go from xi to g)
        return so3_log(A)
    
    def phi(self, in1, in2): # left group action
        return in1.T @ in2
    def psi(self, in1, in2): # right group action
        return in1.T @ in2
    
    def left_mult(self, X, A): # left multiplication
        return A@X
    def right_mult(self,X, B): # right multiplication
        return X@B
    
    def right_inverse(self,A):
        return A.T @ np.linalg.inv(A@A.T)
    
    def theta(self,e): # local coordinate chart for S2
        e1 = np.array([0,0,1]).T # origin   
        cross = np.cross(e1, e)
        dot = e1.T @ e
        arr = np.array([[0,1,0],[0,0,1]])
        normalization = cross / np.absolute(cross)
        t = -np.atan2(cross, dot) * arr * normalization
        return t
    
    def theta_inv(self, epsilon):
        e1 = np.array([0,0,1]).T
        ewedge = so3_wedge(epsilon)
        tinv = self.phi(ewedge, e1)
        return tinv

    def predict(self, u, dt): # u is the angular velocity measurement
        Omega = so3_wedge(u)
        # update xhat 
        self.Xhat = so3_exp(Omega * dt) @ self.Xhat + self.Delta @ self.Xhat
        # update covariance
        A = np.zeros((2,2)) # ---------------make sure this is right
        #R = self.X[0:2, 0:2] # rotation matrix in SO(3)
        R = self.X
        B = np.array([[0,1,0],[0,0,1]]) @ R # 2 X 3
        M = B @ self.Q**3 @ B.T + B @ self.Q @ B.T # 2 x 2
        sigma_dot = A @ self.sigma + self.sigma @ A.T + M # 2x2
        self.sigma += sigma_dot * dt
    
    def update(self, g): #g is the gravity measurement 
        ghat = self.Xhat @ self.xi0
        # innovation
        gres = g - ghat
        # compute state and output gain matrices
        R = self.X # rotation matrix in SO(3)

        arr = np.array([[0, 0],[1,0],[0,1]])
        # kalman gain
        self.Cts = 0.5 * (gres) @ R.T @ arr # equivariamt outpu matrix
        
        Xinv = np.linalg.inv(self.X)
        # define the global errror:
        e = self.phi(Xinv, g)
        # convert into local coordinates
        local_e = self.theta(e)
        E = np.linalg.inv(self.X) @ self.X # group error element

        D = -so3_wedge(self.xi0) # group action derivative

        self.Delta = np.linalg.pinv(D) @ self.theta_inv(D)


Define functions to generate ra/dec data for testing

In [10]:
## define a function that creates fake Ra/Dec vectors to test out the Eqf. (from gemini)
import numpy as np

def starcircle(ra_center, dec_center, radius_deg, num_points=100):
    """
    Generates RA/Dec points for a circle of a given radius around a center point.
    
    Parameters:
    ra_center: Center Right Ascension in degrees (0 to 360)
    dec_center: Center Declination in degrees (-90 to 90)
    radius_deg: Radius of the circle in degrees
    num_points: Number of points to return along the edge
    
    Returns:
    ra_points, dec_points: Two numpy arrays containing the coordinates
    """
    
    # Convert inputs to radians
    ra0 = np.radians(ra_center)
    dec0 = np.radians(dec_center)
    r = np.radians(radius_deg)
    
    # 1. Create a circle of points around the North Pole (Z-axis)
    # At the pole, all points have Dec = 90 - radius
    phi = np.linspace(0, 2 * np.pi, num_points)
    
    # Cartesian coordinates of the circle at the pole
    # x = sin(r)cos(phi), y = sin(r)sin(phi), z = cos(r)
    x = np.sin(r) * np.cos(phi)
    y = np.sin(r) * np.sin(phi)
    z = np.cos(r) * np.ones_like(phi)
    
    # 2. Rotate the circle to the target Declination (rotate around Y axis)
    # We rotate by (90 - dec_center)
    pitch = np.pi/2 - dec0
    x_new = x * np.cos(pitch) + z * np.sin(pitch)
    y_new = y
    z_new = -x * np.sin(pitch) + z * np.cos(pitch)
    
    # 3. Rotate the circle to the target RA (rotate around Z axis)
    ra_final = np.arctan2(y_new, x_new) + ra0
    dec_final = np.arcsin(z_new)
    
    # Convert back to degrees and normalize RA to [0, 360]
    ra_deg = np.degrees(ra_final) % 360
    dec_deg = np.degrees(dec_final)
    
    return ra_deg, dec_deg

Get rotation matrices

In [44]:
import numpy as np
import sympy as sp
from astropy.coordinates import SkyCoord
import astropy.units as units

# create initial rotation based o solved image
# roll is rotation of the camera around its own axis
def initial_rotation(ra, dec, roll):
    # coordinate object for the center of the image
    c = SkyCoord(ra=ra*units.degree,dec = dec*units.degree, frame = 'icrs')

    ra_rad = np.radians(ra)
    dec_rad = np.radians(dec)

    z_b_eci = np.array([
        np.cos(dec_rad)* np.cos(ra_rad),
        np.cos(dec_rad)* np.sin(ra_rad),
        np.sin(dec_rad)
    ])
    # celestial north pole
    n = np.array([0,0,1])

    # body x axis
    x_b_eci = np.cross(n, z_b_eci) 
    x_b_eci /= np.linalg.norm(x_b_eci)

    y_b_eci = np.cross(z_b_eci, x_b_eci)
    
    # DCM to map camera to sky frame 
    R_cam_to_eci = np.column_stack((x_b_eci, y_b_eci, z_b_eci))
    
    # if camera is rotated, rotate around the local Z-axis
    if roll != 0:
        gamma = np.radians(roll)
        R_roll = np.array([
            [np.cos(gamma), -np.sin(gamma), 0],
            [np.sin(gamma),  np.cos(gamma), 0],
            [0,              0,             1]
        ])
        R_cam_to_eci = R_cam_to_eci @ R_roll
        
    return R_cam_to_eci

def world_gravity(Xhat, gb, gmst_deg):
    # rotate gravity from body to ECI (stars)
    g_eci = Xhat @ gb

    # eart shpin rotation matirx
    theta = np.radians(gmst_deg)
    R_stars_to_ecef = np.array([
        [np.cos(theta), np.sin(theta), 0],
        [-np.sin(theta), np.cos(theta), 0],
        [0, 0, 1]
    ])

    # return gravity in the world frame
    gw = R_stars_to_ecef @ g_eci
    return gw

# now, the gravity estimates are in the world frame
# incorporate the ra, dec coordinates to get a position estimation.

def get_latlon(gw):
    gx = gw[0]
    gy = gw[1]
    gz = gw[2]

    # horizontal magnitude
    hz = np.sqrt(gx**2 + gy**2)

    lon = np.rad2deg(np.atan2(-gy, -gx))
    lat = np.rad2deg(np.atan2(-gz, hz))

    return lat, lon

Call the EqF and use it on the star data to get lat/lon

In [67]:
dt = .01
T = 10 # total time
steps = int(T/dt)

eqf = Equivariant(dt, sigma_x=.01, sigma_y=0.01, sigma_z = .01, sigma_gravity = .01)

# FOR NOW - CREATE RA AND DEC VECTORS TO USE (before the plate solver is integrated)
#ra_solved, dec_solved = starcircle(11.6628889, 43.4801667, 10, 100)
ra_solved = 11.6628889
dec_solved = 43.4801667

# solve for latlong depending on time, and ra/dec coordinates:
g_measurement = np.zeros((steps,3)) # gravity data
gyro_data = np.zeros((steps,3)) # gyroscope ANGULAR VLOECITY  data

X0 = initial_rotation(ra_solved, dec_solved,0)
Xhat_storage = [X0.copy()]

time = 830684503.0 # april 29 in J2000 utc

gmst_deg = .004167 * (24110.54841 + 8640184.812866 * time + .093104*(time**2) - (6.2e-6)*(time**3) )
gw = np.zeros((steps, 3))

# filtering loop
for k in range(steps):
    g = g_measurement[k, :]
    omega = gyro_data[k, :]

    eqf.predict(omega, dt)
    eqf.update(g)
    
    Xhat_storage.append(eqf.Xhat)
    gw[k, :] = world_gravity(eqf.Xhat, g, gmst_deg)

lat, lon = get_latlon(gw)

print("Latitude: ", lat,"\n")
print("Longitude: ", lon)

Latitude:  [-0. -0. -0.] 

Longitude:  [-180. -180. -180.]


C:\Users\selen\AppData\Local\Temp\ipykernel_19860\3838132224.py:53: RuntimeWarning: invalid value encountered in divide
  normalization = cross / np.absolute(cross)
